In [ ]:
"""
Calcular medias por fecha para varias columnas a partir de una lista de archivos CSV.
- Sólo se procesan archivos que contienen la columna temporal 'datetime'
- Se ignoran filas con datetime no válidos
- Para cada columna solicitada se crea una columna de salida con sufijo _mean
- El CSV resultante contiene únicamente las columnas de media y el índice datetime

Requisitos:
  pandas
"""

from pathlib import Path
import os
import pandas as pd
from typing import List, Optional

def compute_means_from_filelist(
    file_paths: List[str],
    columns: List[str],
    output_path: str = "media.csv",
    time_column: str = "datetime",
    parse_dates: bool = True,
    required_min_files: int = 1
) -> pd.DataFrame:
    """
    Parámetros
    - file_paths: lista de rutas a los CSV a procesar
    - columns: lista de nombres de columna para los que se calculará la media
    - output_path: ruta del archivo CSV de salida que contendrá solo las medias
    - time_column: nombre de la columna temporal exacto en los CSV
    - parse_dates: si True intenta convertir time_column a tipo fecha
    - required_min_files: número mínimo de archivos válidos necesarios

    Retorna
    - DataFrame resultante con índice datetime y columnas <col>_mean
    """
    # Normalizar y comprobar rutas
    resolved = []
    for p in file_paths:
        ppath = Path(p).expanduser()
        if not ppath.exists():
            print(f"Aviso archivo no encontrado {p}")
            continue
        resolved.append(str(ppath))

    if not resolved:
        raise FileNotFoundError("No se localizaron archivos válidos en la lista proporcionada")

    # Contenedores para las series por variable
    per_variable_series = {col: [] for col in columns}
    valid_file_count = 0

    for fpath in resolved:
        try:
            df = pd.read_csv(fpath)
        except Exception as e:
            print(f"Aviso fallo lectura {fpath} Ignorando este archivo. Error: {e}")
            continue

        # Omitir archivos que no tienen la columna temporal
        if time_column not in df.columns:
            print(f"Aviso {os.path.basename(fpath)} no contiene la columna temporal {time_column}. Archivo omitido.")
            continue

        # Parsear fechas y eliminar filas con datetime no válido
        if parse_dates:
            df[time_column] = pd.to_datetime(df[time_column], errors="coerce")
        df = df.dropna(subset=[time_column])
        if df.shape[0] == 0:
            print(f"Aviso {os.path.basename(fpath)} no tiene filas con {time_column} válido. Archivo omitido.")
            continue

        # Establecer índice temporal
        df = df.set_index(time_column)

        # Por cada columna solicitada, añadir la serie si existe
        basename = os.path.basename(fpath)
        found_any_column = False
        for col in columns:
            if col in df.columns:
                # Forzar tipo numérico; valores no convertibles serán NaN
                s = pd.to_numeric(df[col], errors="coerce")
                s.name = f"{basename}__{col}"
                per_variable_series[col].append(s)
                found_any_column = True

        if found_any_column:
            valid_file_count += 1
        else:
            print(f"Aviso {basename} no contiene ninguna de las columnas solicitadas {columns}. Archivo omitido.")

    if valid_file_count < required_min_files:
        raise ValueError(f"Solo {valid_file_count} archivos válidos. Se requieren al menos {required_min_files}.")

    # Para cada variable calcular la media alineando por índice union
    mean_frames = []
    for col, series_list in per_variable_series.items():
        if not series_list:
            print(f"Aviso no se encontraron datos para la variable {col} en los archivos procesados")
            continue
        aligned = pd.concat(series_list, axis=1, join="outer")
        mean_series = aligned.mean(axis=1, skipna=True)
        mean_series.name = f"{col}_mean"
        mean_frames.append(mean_series)

    if not mean_frames:
        raise ValueError("No se pudieron calcular medias para ninguna de las columnas solicitadas")

    # Unir todas las series de media en un DataFrame final
    final_df = pd.concat(mean_frames, axis=1, join="outer")

    # Guardar resultado con índice datetime
    out_path = Path(output_path).expanduser()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    final_df.to_csv(out_path, index=True)
    print(f"Salida guardada en {out_path} Filas {final_df.shape[0]} Columnas {final_df.shape[1]}")

    return final_df

# Ejemplo de uso
if __name__ == "__main__":
    archivos = [
        "/ruta/a/carpeta1/estacion_A.csv",
        "/ruta/a/otra_carpeta/estacion_B.csv",
        "/ruta/a/estacion_C.csv"
    ]
    columnas_a_promediar = ["x", "y", "z"]
    # Cambie la ruta de salida por la que prefiera antes de ejecutar
    ruta_salida = "media.csv"

    df_result = compute_means_from_filelist(
        file_paths=archivos,
        columns=columnas_a_promediar,
        output_path=ruta_salida,
        time_column="datetime",
        parse_dates=True,
        required_min_files=1
    )
